# NB15 — Can satellite predictors distinguish Musanga abundance?
**Standalone, cached-data-only diagnostic.** Uses exactly the frozen 14,454 × 50 m UAV reference table and optional full-data LOFO HGB predictions in `CLEAN_NB06`. No GEE calls; no retraining.

**Primary question:** Do satellite-near-identical **geographically distinct** pixels exhibit conflicting Musanga fractions? Analyze within-MOS and cross-forest satellite neighbours for annual S2, three-year quarterly S2, annual Earth embeddings and their combined feature vectors.

The audit will:
1. Extract all four feature sets, with no target leakage in predictor-space scaling.
2. Build **site-balanced**, label-blind robust-scaled PCA coordinates separately for each feature family (distances normalized by PCA components' variance). This defines a **relative similarity proxy**, not a physical metric or proven sensor-resolution limit.
3. Identify each cell's nearest eligible neighbour within the **same forest** at least **150 m away**, and the nearest neighbour in **other forests**. Within the same forest, spatially adjacent overlapping/near-identical cells are excluded; cross-forest neighbours are automatically independent geographically. The four pairs are computed on identical reference cells.
4. Summarize the UAV-reference abundance difference (`|Δcover|`) as a function of satellite predictor distance; identify especially contradictory close pairs and high-versus-low (≥40% vs ≤10%) look-alikes, without selecting models by held-out scores.
5. Report prevalence of look-alikes separately by forest, and preserve full pair tables with cell keys for interpretation on UAV mosaics.

**Interpret carefully:** very similar PCA positions may still have discriminative information in discarded dimensions. This is a screening diagnostic, **not** proof that imagery cannot resolve the species. Satellite-time/UAV-time consistency and UAV annotation uncertainty should be reviewed for selected pairs before ecological interpretation. Comparing nearest neighbours in differing feature spaces can be misleading without examining distance distributions and availability of close neighbours; plots include both similarity and cover differences.

Run the notebook with Google Drive mounted. Outputs: `CLEAN_NB06/15_predictor_identifiability/`.

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler,StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from IPython.display import display
import time
roots=[Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/CLEAN_NB06'),
       Path('/content/drive/MyDrive/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/CLEAN_NB06')]
if not any((r/'frozen_50m_reference.parquet').exists() for r in roots):
 from google.colab import drive
 drive.mount('/content/drive')
found=[r for r in roots if (r/'frozen_50m_reference.parquet').exists()]
assert len(found)==1,found
R=found[0];out=R/'15_predictor_identifiability';out.mkdir(parents=True,exist_ok=True)
d=pd.read_parquet(R/'frozen_50m_reference.parquet').reset_index(drop=True)
assert len(d)==14454 and not d.duplicated(['site','mosaic','row','col']).any()
site=d.site.to_numpy();y=d.target_fraction.to_numpy(float)
xy=d[['x','y']].to_numpy(float)
assert np.isfinite(y).all() and np.isfinite(xy).all()
families={
 'S2_annual':[c for c in d if c.startswith('RAW_ANNUAL__')],
 'S2_quarterly_3yr':[c for c in d if c.startswith('S2_QUARTERLY_3YR__')],
 'Earth_embeddings':[c for c in d if c.startswith('EMB_ANNUAL__')],
 'All_123':[c for c in d if c.startswith(('RAW_ANNUAL__','S2_QUARTERLY_3YR__','EMB_ANNUAL__'))],
}
print('Predictor counts:',{k:len(v) for k,v in families.items()})
assert [len(families[k]) for k in families]==[15,44,64,123]
# Use a label-blind *site-balanced* PCA training sample; explicit PCA-space approximation.
rng=np.random.default_rng(20261027)
balance=np.concatenate([rng.choice(np.flatnonzero(site==s),min(480,int(np.sum(site==s))),
                                  replace=False) for s in np.unique(site)])
# Neighbor candidates: query 100 nearest. If no eligible within-site at least 150m away,
# expand query to the entire site; cross-site candidates are obtained from a separate index.
def eligible_neighbor(z,mode,k=100):
 result=np.full(len(d),-1,int);distance=np.full(len(d),np.nan,float)
 for forest in np.unique(site):
  queries=np.flatnonzero(site==forest)
  pool=np.flatnonzero(site==forest) if mode=='within' else np.flatnonzero(site!=forest)
  if not len(pool):continue
  index=NearestNeighbors(n_neighbors=min(k,len(pool)),algorithm='auto',n_jobs=-1).fit(z[pool])
  dd,nn=index.kneighbors(z[queries])
  for j,q in enumerate(queries):
   cand=pool[nn[j]]
   if mode=='within':
    eligible=np.hypot(xy[cand,0]-xy[q,0],xy[cand,1]-xy[q,1])>=150
   else:eligible=np.ones(len(cand),bool)
   if eligible.any():
    first=np.flatnonzero(eligible)[0]
    result[q]=int(cand[first]);distance[q]=float(dd[j,first])
   elif mode=='within':
    # Rare failure due to geographically clustered points; exact full-site fallback.
    full=(np.hypot(xy[pool,0]-xy[q,0],xy[pool,1]-xy[q,1])>=150)
    if full.any():
     ddall=np.linalg.norm(z[pool[full]]-z[q],axis=1)
     at=int(np.argmin(ddall));result[q]=int(pool[full][at]);distance[q]=float(ddall[at])
 return result,distance
allrows=[]
for family,columns in families.items():
 start=time.monotonic()
 X=d[columns].to_numpy(float)
 assert np.isfinite(X).all(),f'Nonfinite predictor values in {family}'
 scaler=RobustScaler(quantile_range=(10,90)).fit(X[balance])
 normalized=np.clip(scaler.transform(X),-10,10)
 # Whitening makes PCA distance measured in approximately equalized latent units.
 nc=min(12,len(columns),len(balance)-1)
 pca=PCA(n_components=nc,whiten=True,random_state=20261027).fit(normalized[balance])
 z=pca.transform(normalized)
 z=np.clip(z,-15,15)
 print('START',family,'components',nc,'cumulative balanced PCA variance',
       round(float(pca.explained_variance_ratio_.sum()),3),flush=True)
 for mode in ['within','cross']:
  neighbor,dist=eligible_neighbor(z,mode)
  valid=neighbor>=0
  for i in np.flatnonzero(valid):
   j=neighbor[i]
   allrows.append(dict(feature_set=family,comparison=mode,index=int(i),
     nearest_index=int(j),site=site[i],neighbor_site=site[j],
     mosaic=d.mosaic.iloc[i],row=int(d.row.iloc[i]),col=int(d.col.iloc[i]),
     neighbor_mosaic=d.mosaic.iloc[j],neighbor_row=int(d.row.iloc[j]),neighbor_col=int(d.col.iloc[j]),
     observed_cover=float(y[i]),neighbor_cover=float(y[j]),
     difference_pp=float(100*abs(y[i]-y[j])),
     distance=float(dist[i]),high_vs_low=bool((y[i]>=.4 and y[j]<=.1) or (y[j]>=.4 and y[i]<=.1)),
     physical_separation_m=float(np.hypot(*(xy[i]-xy[j])))))
 print('DONE',family,f'{time.monotonic()-start:.1f}s',flush=True)
res=pd.DataFrame(allrows)
res.to_parquet(out/'15_nearest_neighbor_pair_audit.parquet',index=False)
# Define the closest 10% within each family/comparison: RELATIVE close, not identical spectra.
res['close_tenth']=False
for (family,mode),indices in res.groupby(['feature_set','comparison']).groups.items():
 threshold=res.loc[indices,'distance'].quantile(.10)
 res.loc[indices,'close_tenth']=res.loc[indices,'distance']<=threshold
summary=(res.groupby(['feature_set','comparison','site'],as_index=False)
 .agg(n=('difference_pp','size'),median_cover_gap_pp=('difference_pp','median'),
 mean_cover_gap_pp=('difference_pp','mean'),high_vs_low_pct=('high_vs_low',lambda x:100*x.mean()),
 distance_p10=('distance',lambda x:x.quantile(.1)),
 distance_median=('distance','median')))
close=(res[res.close_tenth].groupby(['feature_set','comparison','site'])
 .agg(close_n=('difference_pp','size'),close_median_gap_pp=('difference_pp','median'),
      close_high_vs_low_pct=('high_vs_low',lambda x:100*x.mean())).reset_index())
summary=summary.merge(close,on=['feature_set','comparison','site'],how='left')
summary.to_csv(out/'15_predictor_identifiability_summary.csv',index=False)
res.to_parquet(out/'15_nearest_neighbor_pair_audit.parquet',index=False)
print('SUMMARY: nearest-neighbour UAV cover disagreements')
display(summary.round(3))
# Export actual contradictory close pairs to inspect in orthomosaics.
examples=res[res.close_tenth].sort_values(['high_vs_low','difference_pp'],ascending=[False,False])
examples.head(250).to_csv(out/'15_top250_close_lookalikes.csv',index=False)
fig,axs=plt.subplots(2,2,figsize=(12,9),sharex=False,sharey=True)
for ax,(family,g) in zip(axs.flat,res[(res.comparison=='within')&(res.site=='MOS')].groupby('feature_set',sort=False)):
 subset=g.sample(min(len(g),4000),random_state=20261027)
 ax.scatter(subset.distance,subset.difference_pp,s=6,alpha=.17,rasterized=True)
 ax.axhline(30,color='gray',linestyle='--',linewidth=.9)
 ax.set(title=family,xlabel='Relative latent predictor distance',ylabel='Absolute UAV cover difference (pp)',ylim=(0,100))
 ax.grid(alpha=.15)
fig.suptitle('MOS: different Musanga cover among spatially separated satellite-nearest neighbours',fontsize=13)
fig.tight_layout();fig.savefig(out/'15_MOS_similarity_vs_abundance_gap.png',dpi=180);plt.show()
print('BLOCK 15 FINISHED: outputs',out)
print('NOTE: PCA-nearest neighbours are a diagnostic; they do not define an absolute sensor information limit.')
